# React Bookstore with Shopify Storefront API

Build a complete online bookstore using React and Shopify's Storefront API to sell books.

## What You'll Learn:
- Shopify Storefront API integration
- GraphQL queries for products
- Shopping cart with Shopify Checkout
- Product catalog for books
- Real payment processing through Shopify
- Inventory management

## Prerequisites:
- Shopify Partner account (free at https://partners.shopify.com)
- Development store created
- Storefront API access token
- Node.js installed
- Basic React knowledge

Run cells one by one from top to bottom.


## 1. Shopify Setup Instructions

Before coding, set up your Shopify store.


In [ ]:
/*
SHOPIFY SETUP STEPS:

1. Create Shopify Partner Account:
   - Go to https://partners.shopify.com
   - Sign up for free
   - Create a development store

2. Create Development Store:
   - In Partner Dashboard > Stores
   - Click 'Add store' > 'Development store'
   - Fill in store details
   - Choose 'Developer Preview' for latest features

3. Enable Storefront API:
   - Go to your store admin
   - Apps > Develop apps
   - Click 'Allow custom app development'
   - Create an app (e.g., 'Bookstore Frontend')

4. Configure API Scopes:
   - In your app > Configuration
   - Storefront API access scopes:
     ✓ unauthenticated_read_product_listings
     ✓ unauthenticated_read_product_inventory
     ✓ unauthenticated_read_checkouts
     ✓ unauthenticated_write_checkouts
     ✓ unauthenticated_read_customers
   - Save configuration

5. Get API Credentials:
   - Install the app
   - Go to API credentials tab
   - Copy 'Storefront API access token'
   - Note your store domain (e.g., your-store.myshopify.com)

6. Add Book Products:
   - Products > Add product
   - Add book details (title, author, description, price, ISBN)
   - Upload book cover images
   - Set inventory
   - Make products available to 'Online Store' sales channel
*/

console.log('Shopify store setup complete!');


## 2. Project Setup and Dependencies

Install required packages for Shopify integration.


In [ ]:
// Run in terminal:
// npx create-react-app shopify-bookstore
// cd shopify-bookstore
// npm install shopify-buy @apollo/client graphql
// npm install react-router-dom

/*
DEPENDENCIES:
- shopify-buy: Shopify JavaScript Buy SDK
- @apollo/client: GraphQL client for Storefront API
- graphql: GraphQL query language
- react-router-dom: Routing for React
*/

console.log('Dependencies ready!');


## 3. Environment Configuration

Set up environment variables for Shopify credentials.


In [ ]:
// .env file (create in project root)
/*
REACT_APP_SHOPIFY_DOMAIN=your-store.myshopify.com
REACT_APP_SHOPIFY_STOREFRONT_ACCESS_TOKEN=your_storefront_access_token_here
*/

// IMPORTANT: Add .env to .gitignore
// Never commit API tokens to version control!

console.log('Environment configured!');


## 4. Shopify Client Setup

Initialize the Shopify Storefront API client.


In [ ]:
// shopifyClient.js
import Client from 'shopify-buy';

const client = Client.buildClient({
  domain: process.env.REACT_APP_SHOPIFY_DOMAIN,
  storefrontAccessToken: process.env.REACT_APP_SHOPIFY_STOREFRONT_ACCESS_TOKEN
});

export default client;


## 5. Apollo Client Setup (Alternative GraphQL Approach)

Set up Apollo Client for more control over GraphQL queries.


In [ ]:
// apolloClient.js
import { ApolloClient, InMemoryCache, createHttpLink } from '@apollo/client';
import { setContext } from '@apollo/client/link/context';

const httpLink = createHttpLink({
  uri: `https://${process.env.REACT_APP_SHOPIFY_DOMAIN}/api/2024-01/graphql.json`,
});

const middlewareLink = setContext(() => ({
  headers: {
    'X-Shopify-Storefront-Access-Token': process.env.REACT_APP_SHOPIFY_STOREFRONT_ACCESS_TOKEN,
    'Content-Type': 'application/json',
  },
}));

const apolloClient = new ApolloClient({
  link: middlewareLink.concat(httpLink),
  cache: new InMemoryCache(),
});

export default apolloClient;


## 6. GraphQL Queries for Books

Define GraphQL queries to fetch book products from Shopify.


In [ ]:
// queries.js
import { gql } from '@apollo/client';

// Get all products (books)
export const GET_PRODUCTS = gql`
  query GetProducts($first: Int!) {
    products(first: $first) {
      edges {
        node {
          id
          title
          description
          handle
          priceRange {
            minVariantPrice {
              amount
              currencyCode
            }
          }
          images(first: 1) {
            edges {
              node {
                url
                altText
              }
            }
          }
          variants(first: 1) {
            edges {
              node {
                id
                title
                priceV2 {
                  amount
                  currencyCode
                }
                availableForSale
              }
            }
          }
          metafields(identifiers: [
            { namespace: "custom", key: "author" },
            { namespace: "custom", key: "isbn" },
            { namespace: "custom", key: "publisher" },
            { namespace: "custom", key: "pages" }
          ]) {
            key
            value
          }
        }
      }
    }
  }
`;

// Get single product by handle
export const GET_PRODUCT_BY_HANDLE = gql`
  query GetProductByHandle($handle: String!) {
    productByHandle(handle: $handle) {
      id
      title
      description
      descriptionHtml
      priceRange {
        minVariantPrice {
          amount
          currencyCode
        }
      }
      images(first: 5) {
        edges {
          node {
            url
            altText
          }
        }
      }
      variants(first: 10) {
        edges {
          node {
            id
            title
            priceV2 {
              amount
              currencyCode
            }
            availableForSale
            quantityAvailable
          }
        }
      }
      metafields(identifiers: [
        { namespace: "custom", key: "author" },
        { namespace: "custom", key: "isbn" },
        { namespace: "custom", key: "publisher" },
        { namespace: "custom", key: "pages" },
        { namespace: "custom", key: "publication_date" }
      ]) {
        key
        value
      }
    }
  }
`;

// Create checkout
export const CREATE_CHECKOUT = gql`
  mutation CreateCheckout($input: CheckoutCreateInput!) {
    checkoutCreate(input: $input) {
      checkout {
        id
        webUrl
        lineItems(first: 10) {
          edges {
            node {
              id
              title
              quantity
            }
          }
        }
      }
      checkoutUserErrors {
        message
        field
      }
    }
  }
`;

// Add items to checkout
export const CHECKOUT_LINE_ITEMS_ADD = gql`
  mutation CheckoutLineItemsAdd($checkoutId: ID!, $lineItems: [CheckoutLineItemInput!]!) {
    checkoutLineItemsAdd(checkoutId: $checkoutId, lineItems: $lineItems) {
      checkout {
        id
        webUrl
        lineItems(first: 10) {
          edges {
            node {
              id
              title
              quantity
              variant {
                priceV2 {
                  amount
                  currencyCode
                }
              }
            }
          }
        }
        totalPriceV2 {
          amount
          currencyCode
        }
      }
      checkoutUserErrors {
        message
        field
      }
    }
  }
`;


## 7. Shopping Cart Context

Create a context to manage cart state with Shopify checkout.


In [ ]:
// CartContext.js
import React, { createContext, useContext, useState, useEffect } from 'react';
import shopifyClient from './shopifyClient';

const CartContext = createContext();

export const CartProvider = ({ children }) => {
  const [checkout, setCheckout] = useState(null);
  const [isLoading, setIsLoading] = useState(false);
  
  // Initialize checkout on mount
  useEffect(() => {
    initializeCheckout();
  }, []);
  
  const initializeCheckout = async () => {
    try {
      // Check if checkout exists in localStorage
      const checkoutId = localStorage.getItem('checkoutId');
      
      if (checkoutId) {
        // Fetch existing checkout
        const existingCheckout = await shopifyClient.checkout.fetch(checkoutId);
        
        // Check if checkout is completed
        if (!existingCheckout.completedAt) {
          setCheckout(existingCheckout);
          return;
        }
      }
      
      // Create new checkout
      const newCheckout = await shopifyClient.checkout.create();
      setCheckout(newCheckout);
      localStorage.setItem('checkoutId', newCheckout.id);
    } catch (error) {
      console.error('Error initializing checkout:', error);
    }
  };
  
  const addToCart = async (variantId, quantity = 1) => {
    setIsLoading(true);
    try {
      const lineItemsToAdd = [{ variantId, quantity }];
      const updatedCheckout = await shopifyClient.checkout.addLineItems(
        checkout.id,
        lineItemsToAdd
      );
      setCheckout(updatedCheckout);
      return updatedCheckout;
    } catch (error) {
      console.error('Error adding to cart:', error);
      throw error;
    } finally {
      setIsLoading(false);
    }
  };
  
  const updateLineItem = async (lineItemId, quantity) => {
    setIsLoading(true);
    try {
      const lineItemsToUpdate = [{ id: lineItemId, quantity }];
      const updatedCheckout = await shopifyClient.checkout.updateLineItems(
        checkout.id,
        lineItemsToUpdate
      );
      setCheckout(updatedCheckout);
    } catch (error) {
      console.error('Error updating line item:', error);
      throw error;
    } finally {
      setIsLoading(false);
    }
  };
  
  const removeLineItem = async (lineItemId) => {
    setIsLoading(true);
    try {
      const updatedCheckout = await shopifyClient.checkout.removeLineItems(
        checkout.id,
        [lineItemId]
      );
      setCheckout(updatedCheckout);
    } catch (error) {
      console.error('Error removing line item:', error);
      throw error;
    } finally {
      setIsLoading(false);
    }
  };
  
  const getCartCount = () => {
    if (!checkout || !checkout.lineItems) return 0;
    return checkout.lineItems.reduce((total, item) => total + item.quantity, 0);
  };
  
  const getCartTotal = () => {
    if (!checkout) return '0.00';
    return checkout.totalPrice;
  };
  
  return (
    <CartContext.Provider value={{
      checkout,
      isLoading,
      addToCart,
      updateLineItem,
      removeLineItem,
      getCartCount,
      getCartTotal
    }}>
      {children}
    </CartContext.Provider>
  );
};

export const useCart = () => {
  const context = useContext(CartContext);
  if (!context) {
    throw new Error('useCart must be used within CartProvider');
  }
  return context;
};


## 8. Book Card Component

Display individual book products.


In [ ]:
// BookCard.js
import React from 'react';
import { useCart } from './CartContext';
import { Link } from 'react-router-dom';

const BookCard = ({ product }) => {
  const { addToCart, isLoading } = useCart();
  
  const image = product.images?.edges[0]?.node?.url || 'https://via.placeholder.com/300x400?text=No+Image';
  const price = product.priceRange?.minVariantPrice?.amount || '0.00';
  const currency = product.priceRange?.minVariantPrice?.currencyCode || 'USD';
  const variantId = product.variants?.edges[0]?.node?.id;
  const available = product.variants?.edges[0]?.node?.availableForSale;
  
  // Extract metafields
  const author = product.metafields?.find(m => m.key === 'author')?.value || 'Unknown Author';
  
  const handleAddToCart = async (e) => {
    e.preventDefault();
    if (!variantId || !available) return;
    
    try {
      await addToCart(variantId, 1);
      alert(`${product.title} added to cart!`);
    } catch (error) {
      alert('Failed to add to cart. Please try again.');
    }
  };
  
  return (
    <div style={{
      border: '1px solid #e0e0e0',
      borderRadius: '8px',
      overflow: 'hidden',
      transition: 'transform 0.2s, box-shadow 0.2s',
      cursor: 'pointer',
      backgroundColor: 'white'
    }}
    onMouseEnter={(e) => {
      e.currentTarget.style.transform = 'translateY(-5px)';
      e.currentTarget.style.boxShadow = '0 4px 12px rgba(0,0,0,0.15)';
    }}
    onMouseLeave={(e) => {
      e.currentTarget.style.transform = 'translateY(0)';
      e.currentTarget.style.boxShadow = 'none';
    }}>
      <Link to={`/book/${product.handle}`} style={{ textDecoration: 'none', color: 'inherit' }}>
        <img 
          src={image} 
          alt={product.title}
          style={{ width: '100%', height: '350px', objectFit: 'cover' }}
        />
        <div style={{ padding: '15px' }}>
          <h3 style={{ 
            margin: '0 0 5px', 
            fontSize: '18px',
            overflow: 'hidden',
            textOverflow: 'ellipsis',
            whiteSpace: 'nowrap'
          }}>
            {product.title}
          </h3>
          <p style={{ margin: '0 0 10px', color: '#666', fontSize: '14px' }}>
            by {author}
          </p>
          <p style={{ 
            fontSize: '20px', 
            fontWeight: 'bold', 
            color: '#2c3e50',
            margin: '10px 0'
          }}>
            {new Intl.NumberFormat('en-US', {
              style: 'currency',
              currency: currency
            }).format(price)}
          </p>
        </div>
      </Link>
      <div style={{ padding: '0 15px 15px' }}>
        <button
          onClick={handleAddToCart}
          disabled={!available || isLoading}
          style={{
            width: '100%',
            padding: '12px',
            backgroundColor: available ? '#4CAF50' : '#ccc',
            color: 'white',
            border: 'none',
            borderRadius: '4px',
            fontSize: '16px',
            cursor: available ? 'pointer' : 'not-allowed',
            transition: 'background-color 0.2s'
          }}
          onMouseEnter={(e) => available && (e.target.style.backgroundColor = '#45a049')}
          onMouseLeave={(e) => available && (e.target.style.backgroundColor = '#4CAF50')}>
          {available ? 'Add to Cart' : 'Out of Stock'}
        </button>
      </div>
    </div>
  );
};

export default BookCard;


## 9. Book Catalog Component

Display all books using the Shopify Storefront API.


In [ ]:
// BookCatalog.js
import React, { useState, useEffect } from 'react';
import { useQuery } from '@apollo/client';
import { GET_PRODUCTS } from './queries';
import BookCard from './BookCard';

const BookCatalog = () => {
  const [searchTerm, setSearchTerm] = useState('');
  const { loading, error, data } = useQuery(GET_PRODUCTS, {
    variables: { first: 20 }
  });
  
  if (loading) {
    return (
      <div style={{ textAlign: 'center', padding: '50px' }}>
        <h2>Loading books...</h2>
      </div>
    );
  }
  
  if (error) {
    return (
      <div style={{ textAlign: 'center', padding: '50px', color: 'red' }}>
        <h2>Error loading books</h2>
        <p>{error.message}</p>
      </div>
    );
  }
  
  const products = data?.products?.edges?.map(edge => edge.node) || [];
  
  const filteredProducts = products.filter(product =>
    product.title.toLowerCase().includes(searchTerm.toLowerCase()) ||
    product.description?.toLowerCase().includes(searchTerm.toLowerCase())
  );
  
  return (
    <div style={{ padding: '20px', maxWidth: '1400px', margin: '0 auto' }}>
      <div style={{ marginBottom: '30px' }}>
        <h1 style={{ textAlign: 'center', marginBottom: '20px' }}>📚 Our Book Collection</h1>
        <input
          type="text"
          placeholder="Search books by title or description..."
          value={searchTerm}
          onChange={(e) => setSearchTerm(e.target.value)}
          style={{
            width: '100%',
            maxWidth: '600px',
            padding: '12px 20px',
            fontSize: '16px',
            border: '2px solid #ddd',
            borderRadius: '25px',
            display: 'block',
            margin: '0 auto',
            outline: 'none'
          }}
        />
      </div>
      
      {filteredProducts.length === 0 ? (
        <div style={{ textAlign: 'center', padding: '50px' }}>
          <h2>No books found</h2>
          <p>Try a different search term</p>
        </div>
      ) : (
        <>
          <p style={{ textAlign: 'center', color: '#666', marginBottom: '20px' }}>
            Showing {filteredProducts.length} book{filteredProducts.length !== 1 ? 's' : ''}
          </p>
          <div style={{
            display: 'grid',
            gridTemplateColumns: 'repeat(auto-fill, minmax(250px, 1fr))',
            gap: '25px'
          }}>
            {filteredProducts.map(product => (
              <BookCard key={product.id} product={product} />
            ))}
          </div>
        </>
      )}
    </div>
  );
};

export default BookCatalog;


## 10. Book Detail Page

Show detailed information about a single book.


In [ ]:
// BookDetail.js
import React, { useState } from 'react';
import { useParams, useNavigate } from 'react-router-dom';
import { useQuery } from '@apollo/client';
import { GET_PRODUCT_BY_HANDLE } from './queries';
import { useCart } from './CartContext';

const BookDetail = () => {
  const { handle } = useParams();
  const navigate = useNavigate();
  const { addToCart, isLoading: cartLoading } = useCart();
  const [quantity, setQuantity] = useState(1);
  const [selectedImage, setSelectedImage] = useState(0);
  
  const { loading, error, data } = useQuery(GET_PRODUCT_BY_HANDLE, {
    variables: { handle }
  });
  
  if (loading) {
    return <div style={{ textAlign: 'center', padding: '50px' }}><h2>Loading...</h2></div>;
  }
  
  if (error || !data?.productByHandle) {
    return (
      <div style={{ textAlign: 'center', padding: '50px' }}>
        <h2>Book not found</h2>
        <button onClick={() => navigate('/')} style={{ padding: '10px 20px', marginTop: '20px' }}>
          Back to Catalog
        </button>
      </div>
    );
  }
  
  const product = data.productByHandle;
  const images = product.images?.edges?.map(edge => edge.node) || [];
  const variant = product.variants?.edges[0]?.node;
  const price = variant?.priceV2?.amount || '0.00';
  const currency = variant?.priceV2?.currencyCode || 'USD';
  const available = variant?.availableForSale;
  
  // Extract metafields
  const getMetafield = (key) => product.metafields?.find(m => m.key === key)?.value;
  const author = getMetafield('author') || 'Unknown Author';
  const isbn = getMetafield('isbn');
  const publisher = getMetafield('publisher');
  const pages = getMetafield('pages');
  const publicationDate = getMetafield('publication_date');
  
  const handleAddToCart = async () => {
    if (!variant?.id || !available) return;
    
    try {
      await addToCart(variant.id, quantity);
      alert(`${quantity} x ${product.title} added to cart!`);
    } catch (error) {
      alert('Failed to add to cart. Please try again.');
    }
  };
  
  return (
    <div style={{ padding: '20px', maxWidth: '1200px', margin: '0 auto' }}>
      <button 
        onClick={() => navigate('/')}
        style={{
          padding: '10px 20px',
          marginBottom: '20px',
          backgroundColor: '#f0f0f0',
          border: 'none',
          borderRadius: '4px',
          cursor: 'pointer'
        }}>
        ← Back to Catalog
      </button>
      
      <div style={{ display: 'grid', gridTemplateColumns: '1fr 1fr', gap: '40px' }}>
        {/* Images */}
        <div>
          <img 
            src={images[selectedImage]?.url || 'https://via.placeholder.com/500x700?text=No+Image'}
            alt={product.title}
            style={{ width: '100%', borderRadius: '8px', marginBottom: '15px' }}
          />
          {images.length > 1 && (
            <div style={{ display: 'flex', gap: '10px', overflowX: 'auto' }}>
              {images.map((img, idx) => (
                <img
                  key={idx}
                  src={img.url}
                  alt={`${product.title} ${idx + 1}`}
                  onClick={() => setSelectedImage(idx)}
                  style={{
                    width: '80px',
                    height: '100px',
                    objectFit: 'cover',
                    borderRadius: '4px',
                    cursor: 'pointer',
                    border: selectedImage === idx ? '2px solid #4CAF50' : '2px solid transparent'
                  }}
                />
              ))}
            </div>
          )}
        </div>
        
        {/* Details */}
        <div>
          <h1 style={{ marginTop: 0 }}>{product.title}</h1>
          <p style={{ fontSize: '18px', color: '#666', marginBottom: '10px' }}>by {author}</p>
          
          <p style={{ fontSize: '32px', fontWeight: 'bold', color: '#2c3e50', margin: '20px 0' }}>
            {new Intl.NumberFormat('en-US', {
              style: 'currency',
              currency: currency
            }).format(price)}
          </p>
          
          <div style={{ marginBottom: '20px' }}>
            <h3>Book Details</h3>
            {isbn && <p><strong>ISBN:</strong> {isbn}</p>}
            {publisher && <p><strong>Publisher:</strong> {publisher}</p>}
            {pages && <p><strong>Pages:</strong> {pages}</p>}
            {publicationDate && <p><strong>Publication Date:</strong> {publicationDate}</p>}
            <p><strong>Availability:</strong> {available ? '✓ In Stock' : '✗ Out of Stock'}</p>
          </div>
          
          <div style={{ marginBottom: '20px' }}>
            <h3>Description</h3>
            <div dangerouslySetInnerHTML={{ __html: product.descriptionHtml || product.description }} />
          </div>
          
          <div style={{ display: 'flex', gap: '15px', alignItems: 'center', marginTop: '30px' }}>
            <div>
              <label style={{ display: 'block', marginBottom: '5px', fontWeight: 'bold' }}>Quantity:</label>
              <input
                type="number"
                min="1"
                max="10"
                value={quantity}
                onChange={(e) => setQuantity(parseInt(e.target.value) || 1)}
                style={{
                  padding: '10px',
                  fontSize: '16px',
                  width: '80px',
                  border: '1px solid #ddd',
                  borderRadius: '4px'
                }}
              />
            </div>
            <button
              onClick={handleAddToCart}
              disabled={!available || cartLoading}
              style={{
                flex: 1,
                padding: '15px 30px',
                backgroundColor: available ? '#4CAF50' : '#ccc',
                color: 'white',
                border: 'none',
                borderRadius: '4px',
                fontSize: '18px',
                cursor: available ? 'pointer' : 'not-allowed',
                marginTop: '25px'
              }}>
              {available ? 'Add to Cart' : 'Out of Stock'}
            </button>
          </div>
        </div>
      </div>
    </div>
  );
};

export default BookDetail;


## 11. Shopping Cart Component

Display cart items and proceed to Shopify checkout.


In [ ]:
// Cart.js
import React from 'react';
import { useCart } from './CartContext';
import { useNavigate } from 'react-router-dom';

const Cart = () => {
  const { checkout, updateLineItem, removeLineItem, isLoading } = useCart();
  const navigate = useNavigate();
  
  if (!checkout || !checkout.lineItems || checkout.lineItems.length === 0) {
    return (
      <div style={{ textAlign: 'center', padding: '50px' }}>
        <h2>Your cart is empty</h2>
        <p>Browse our collection and add some books!</p>
        <button
          onClick={() => navigate('/')}
          style={{
            padding: '12px 24px',
            backgroundColor: '#4CAF50',
            color: 'white',
            border: 'none',
            borderRadius: '4px',
            fontSize: '16px',
            cursor: 'pointer',
            marginTop: '20px'
          }}>
          Browse Books
        </button>
      </div>
    );
  }
  
  const handleCheckout = () => {
    // Redirect to Shopify checkout
    window.location.href = checkout.webUrl;
  };
  
  return (
    <div style={{ padding: '20px', maxWidth: '900px', margin: '0 auto' }}>
      <h1>Shopping Cart</h1>
      
      <div style={{ marginTop: '30px' }}>
        {checkout.lineItems.map(item => (
          <div key={item.id} style={{
            display: 'grid',
            gridTemplateColumns: '100px 1fr auto',
            gap: '20px',
            padding: '20px',
            borderBottom: '1px solid #e0e0e0',
            alignItems: 'center'
          }}>
            <img 
              src={item.variant.image?.src || 'https://via.placeholder.com/100x140?text=Book'}
              alt={item.title}
              style={{ width: '100%', borderRadius: '4px' }}
            />
            
            <div>
              <h3 style={{ margin: '0 0 5px' }}>{item.title}</h3>
              <p style={{ margin: '0', color: '#666' }}>
                {new Intl.NumberFormat('en-US', {
                  style: 'currency',
                  currency: item.variant.priceV2.currencyCode
                }).format(item.variant.priceV2.amount)}
              </p>
              
              <div style={{ display: 'flex', gap: '10px', alignItems: 'center', marginTop: '10px' }}>
                <label>Quantity:</label>
                <button
                  onClick={() => updateLineItem(item.id, item.quantity - 1)}
                  disabled={isLoading || item.quantity <= 1}
                  style={{
                    padding: '5px 10px',
                    cursor: item.quantity > 1 ? 'pointer' : 'not-allowed'
                  }}>
                  -
                </button>
                <span style={{ minWidth: '30px', textAlign: 'center' }}>{item.quantity}</span>
                <button
                  onClick={() => updateLineItem(item.id, item.quantity + 1)}
                  disabled={isLoading}
                  style={{ padding: '5px 10px', cursor: 'pointer' }}>
                  +
                </button>
                <button
                  onClick={() => removeLineItem(item.id)}
                  disabled={isLoading}
                  style={{
                    marginLeft: '20px',
                    padding: '5px 15px',
                    backgroundColor: '#f44336',
                    color: 'white',
                    border: 'none',
                    borderRadius: '4px',
                    cursor: 'pointer'
                  }}>
                  Remove
                </button>
              </div>
            </div>
            
            <div style={{ textAlign: 'right' }}>
              <p style={{ fontSize: '18px', fontWeight: 'bold', margin: 0 }}>
                {new Intl.NumberFormat('en-US', {
                  style: 'currency',
                  currency: item.variant.priceV2.currencyCode
                }).format(item.variant.priceV2.amount * item.quantity)}
              </p>
            </div>
          </div>
        ))}
      </div>
      
      <div style={{ marginTop: '30px', textAlign: 'right' }}>
        <div style={{ fontSize: '14px', color: '#666', marginBottom: '10px' }}>
          <p>Subtotal: {new Intl.NumberFormat('en-US', {
            style: 'currency',
            currency: checkout.currencyCode
          }).format(checkout.subtotalPrice)}</p>
          <p>Tax: {new Intl.NumberFormat('en-US', {
            style: 'currency',
            currency: checkout.currencyCode
          }).format(checkout.totalTax)}</p>
        </div>
        <h2 style={{ margin: '10px 0 20px' }}>
          Total: {new Intl.NumberFormat('en-US', {
            style: 'currency',
            currency: checkout.currencyCode
          }).format(checkout.totalPrice)}
        </h2>
        
        <div style={{ display: 'flex', gap: '15px', justifyContent: 'flex-end' }}>
          <button
            onClick={() => navigate('/')}
            style={{
              padding: '15px 30px',
              backgroundColor: '#f0f0f0',
              color: '#333',
              border: 'none',
              borderRadius: '4px',
              fontSize: '16px',
              cursor: 'pointer'
            }}>
            Continue Shopping
          </button>
          <button
            onClick={handleCheckout}
            disabled={isLoading}
            style={{
              padding: '15px 30px',
              backgroundColor: '#4CAF50',
              color: 'white',
              border: 'none',
              borderRadius: '4px',
              fontSize: '16px',
              cursor: 'pointer',
              fontWeight: 'bold'
            }}>
            Proceed to Checkout →
          </button>
        </div>
      </div>
    </div>
  );
};

export default Cart;


## 12. Header Component with Cart Badge

Navigation header with cart item count.


In [ ]:
// Header.js
import React from 'react';
import { Link } from 'react-router-dom';
import { useCart } from './CartContext';

const Header = () => {
  const { getCartCount } = useCart();
  const cartCount = getCartCount();
  
  return (
    <header style={{
      backgroundColor: '#2c3e50',
      color: 'white',
      padding: '20px 40px',
      display: 'flex',
      justifyContent: 'space-between',
      alignItems: 'center',
      boxShadow: '0 2px 5px rgba(0,0,0,0.1)'
    }}>
      <Link to="/" style={{ textDecoration: 'none', color: 'white' }}>
        <h1 style={{ margin: 0, display: 'flex', alignItems: 'center', gap: '10px' }}>
          📚 Shopify Bookstore
        </h1>
      </Link>
      
      <nav style={{ display: 'flex', gap: '30px', alignItems: 'center' }}>
        <Link 
          to="/" 
          style={{ 
            color: 'white', 
            textDecoration: 'none',
            fontSize: '16px',
            transition: 'opacity 0.2s'
          }}
          onMouseEnter={(e) => e.target.style.opacity = '0.7'}
          onMouseLeave={(e) => e.target.style.opacity = '1'}>
          Browse Books
        </Link>
        
        <Link 
          to="/cart" 
          style={{ 
            color: 'white', 
            textDecoration: 'none',
            position: 'relative',
            display: 'flex',
            alignItems: 'center',
            gap: '8px',
            backgroundColor: '#4CAF50',
            padding: '10px 20px',
            borderRadius: '4px',
            transition: 'background-color 0.2s'
          }}
          onMouseEnter={(e) => e.currentTarget.style.backgroundColor = '#45a049'}
          onMouseLeave={(e) => e.currentTarget.style.backgroundColor = '#4CAF50'}>
          🛒 Cart
          {cartCount > 0 && (
            <span style={{
              backgroundColor: '#f44336',
              color: 'white',
              borderRadius: '50%',
              width: '24px',
              height: '24px',
              display: 'flex',
              alignItems: 'center',
              justifyContent: 'center',
              fontSize: '12px',
              fontWeight: 'bold'
            }}>
              {cartCount}
            </span>
          )}
        </Link>
      </nav>
    </header>
  );
};

export default Header;


## 13. Main App Component

Bring everything together with routing.


In [ ]:
// App.js
import React from 'react';
import { BrowserRouter as Router, Routes, Route } from 'react-router-dom';
import { ApolloProvider } from '@apollo/client';
import apolloClient from './apolloClient';
import { CartProvider } from './CartContext';
import Header from './Header';
import BookCatalog from './BookCatalog';
import BookDetail from './BookDetail';
import Cart from './Cart';

function App() {
  return (
    <ApolloProvider client={apolloClient}>
      <CartProvider>
        <Router>
          <div style={{ minHeight: '100vh', backgroundColor: '#f5f5f5' }}>
            <Header />
            <Routes>
              <Route path="/" element={<BookCatalog />} />
              <Route path="/book/:handle" element={<BookDetail />} />
              <Route path="/cart" element={<Cart />} />
            </Routes>
            
            <footer style={{
              backgroundColor: '#2c3e50',
              color: 'white',
              textAlign: 'center',
              padding: '30px',
              marginTop: '50px'
            }}>
              <p>© 2024 Shopify Bookstore. Powered by Shopify Storefront API.</p>
            </footer>
          </div>
        </Router>
      </CartProvider>
    </ApolloProvider>
  );
}

export default App;


## 14. Entry Point

React app entry point.


In [ ]:
// index.js
import React from 'react';
import ReactDOM from 'react-dom/client';
import './index.css';
import App from './App';

const root = ReactDOM.createRoot(document.getElementById('root'));
root.render(
  <React.StrictMode>
    <App />
  </React.StrictMode>
);


## 15. Basic Styling

Global CSS styles.


In [ ]:
/* index.css */
* {
  box-sizing: border-box;
}

body {
  margin: 0;
  font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', 'Roboto', 'Oxygen',
    'Ubuntu', 'Cantarell', 'Fira Sans', 'Droid Sans', 'Helvetica Neue',
    sans-serif;
  -webkit-font-smoothing: antialiased;
  -moz-osx-font-smoothing: grayscale;
}

code {
  font-family: source-code-pro, Menlo, Monaco, Consolas, 'Courier New',
    monospace;
}

button {
  font-family: inherit;
}

input {
  font-family: inherit;
}


## 16. Package.json

Complete package configuration.


In [ ]:
// package.json
{
  "name": "shopify-bookstore",
  "version": "1.0.0",
  "description": "Online bookstore powered by Shopify Storefront API",
  "private": true,
  "dependencies": {
    "react": "^18.2.0",
    "react-dom": "^18.2.0",
    "react-router-dom": "^6.20.0",
    "shopify-buy": "^2.17.0",
    "@apollo/client": "^3.8.8",
    "graphql": "^16.8.1"
  },
  "scripts": {
    "start": "react-scripts start",
    "build": "react-scripts build",
    "test": "react-scripts test",
    "eject": "react-scripts eject"
  },
  "devDependencies": {
    "react-scripts": "^5.0.1"
  },
  "eslintConfig": {
    "extends": [
      "react-app"
    ]
  },
  "browserslist": {
    "production": [
      ">0.2%",
      "not dead",
      "not op_mini all"
    ],
    "development": [
      "last 1 chrome version",
      "last 1 firefox version",
      "last 1 safari version"
    ]
  }
}


## 17. Adding Book Metafields in Shopify

How to add custom book information to products.


In [ ]:
/*
ADDING BOOK METAFIELDS IN SHOPIFY:

1. Go to Shopify Admin > Settings > Custom data

2. Click on 'Products' > 'Add definition'

3. Create these metafield definitions:

   Author:
   - Name: Author
   - Namespace and key: custom.author
   - Type: Single line text
   - Storefront access: ✓ Enabled

   ISBN:
   - Name: ISBN
   - Namespace and key: custom.isbn
   - Type: Single line text
   - Storefront access: ✓ Enabled

   Publisher:
   - Name: Publisher
   - Namespace and key: custom.publisher
   - Type: Single line text
   - Storefront access: ✓ Enabled

   Pages:
   - Name: Pages
   - Namespace and key: custom.pages
   - Type: Integer
   - Storefront access: ✓ Enabled

   Publication Date:
   - Name: Publication Date
   - Namespace and key: custom.publication_date
   - Type: Date
   - Storefront access: ✓ Enabled

4. When adding/editing products:
   - Scroll to 'Metafields' section
   - Fill in the book-specific information
   - Save the product

5. These metafields will be accessible via the Storefront API
   and displayed in your React app!
*/

console.log('Metafields configured!');


## 18. Complete Setup Guide

Step-by-step instructions to run your bookstore.


In [ ]:
/*
COMPLETE SETUP GUIDE:

STEP 1: Create Shopify Development Store
1. Go to https://partners.shopify.com
2. Create a Partner account (free)
3. Stores > Add store > Development store
4. Fill in store details and create

STEP 2: Enable Storefront API
1. Go to your store admin
2. Apps > Develop apps > Allow custom app development
3. Create app > Name it 'Bookstore Frontend'
4. Configure > Storefront API scopes:
   - unauthenticated_read_product_listings
   - unauthenticated_read_checkouts
   - unauthenticated_write_checkouts
5. Save > Install app
6. API credentials > Copy Storefront API access token

STEP 3: Add Book Products
1. Products > Add product
2. Add book details:
   - Title (e.g., "The Great Gatsby")
   - Description
   - Price
   - Upload cover image
3. Add metafields (author, ISBN, publisher, pages)
4. Set inventory
5. Make available to 'Online Store' channel
6. Save
7. Repeat for multiple books

STEP 4: Create React Project
npx create-react-app shopify-bookstore
cd shopify-bookstore

STEP 5: Install Dependencies
npm install shopify-buy @apollo/client graphql react-router-dom

STEP 6: Create .env File
Create .env in project root:
REACT_APP_SHOPIFY_DOMAIN=your-store.myshopify.com
REACT_APP_SHOPIFY_STOREFRONT_ACCESS_TOKEN=your_token_here

STEP 7: Create All Component Files
Create files as shown in previous cells:
- shopifyClient.js
- apolloClient.js
- queries.js
- CartContext.js
- BookCard.js
- BookCatalog.js
- BookDetail.js
- Cart.js
- Header.js
- App.js
- index.js
- index.css

STEP 8: Run the Application
npm start

STEP 9: Test the Store
1. Browse books
2. Click on a book to see details
3. Add books to cart
4. View cart
5. Click 'Proceed to Checkout'
6. Complete checkout on Shopify (use test payment)

STEP 10: Test Payment
Use Shopify Bogus Gateway for testing:
- Card: 1 (any number of 1s)
- Expiry: Any future date
- CVV: Any 3 digits
*/

console.log('Setup complete! Your bookstore is ready!');


## 19. Deployment Guide

Deploy your bookstore to production.


In [ ]:
/*
DEPLOYMENT OPTIONS:

1. VERCEL (Recommended):
   - Install Vercel CLI: npm i -g vercel
   - Run: vercel
   - Add environment variables in Vercel dashboard:
     * REACT_APP_SHOPIFY_DOMAIN
     * REACT_APP_SHOPIFY_STOREFRONT_ACCESS_TOKEN
   - Deploy: vercel --prod

2. NETLIFY:
   - Install Netlify CLI: npm i -g netlify-cli
   - Build: npm run build
   - Deploy: netlify deploy --prod --dir=build
   - Add environment variables in Netlify dashboard

3. GITHUB PAGES:
   - Install gh-pages: npm install --save-dev gh-pages
   - Add to package.json:
     "homepage": "https://yourusername.github.io/shopify-bookstore",
     "scripts": {
       "predeploy": "npm run build",
       "deploy": "gh-pages -d build"
     }
   - Deploy: npm run deploy

4. CUSTOM DOMAIN:
   - Purchase domain (Namecheap, Google Domains)
   - Configure DNS in your hosting provider
   - Add custom domain in Vercel/Netlify settings
   - SSL certificate (automatic with Vercel/Netlify)

PRE-DEPLOYMENT CHECKLIST:
□ Test all features locally
□ Verify environment variables are set
□ Test checkout flow end-to-end
□ Check mobile responsiveness
□ Optimize images
□ Test on multiple browsers
□ Set up error tracking (Sentry)
□ Configure analytics (Google Analytics)
□ Test payment processing
□ Review Shopify store settings
*/

console.log('Ready to deploy!');


## 20. Advanced Features to Add

Enhance your bookstore with these features.


In [ ]:
/*
ADVANCED FEATURES:

1. SEARCH & FILTERS:
   - Full-text search
   - Filter by genre/category
   - Filter by price range
   - Filter by author
   - Sort options (price, title, newest)

2. COLLECTIONS:
   - Featured books
   - Bestsellers
   - New arrivals
   - Genre-based collections
   - Author collections

3. PRODUCT REVIEWS:
   - Customer reviews and ratings
   - Review submission form
   - Star ratings display
   - Review moderation

4. WISHLIST:
   - Save books for later
   - Persistent wishlist (localStorage)
   - Share wishlist
   - Move to cart from wishlist

5. USER ACCOUNTS:
   - Customer login/registration
   - Order history
   - Saved addresses
   - Account preferences

6. RECOMMENDATIONS:
   - Related books
   - "Customers also bought"
   - Personalized recommendations
   - Recently viewed books

7. ENHANCED UI:
   - Book preview/sample pages
   - Image zoom on hover
   - Quick view modal
   - Loading skeletons
   - Toast notifications

8. INVENTORY MANAGEMENT:
   - Low stock warnings
   - Out of stock notifications
   - Pre-order functionality
   - Back in stock alerts

9. DISCOUNTS & PROMOTIONS:
   - Discount codes
   - Bundle deals
   - Free shipping thresholds
   - Flash sales

10. ANALYTICS:
    - Google Analytics integration
    - Conversion tracking
    - Product view tracking
    - Cart abandonment tracking

11. EMAIL MARKETING:
    - Newsletter signup
    - Abandoned cart emails
    - Order confirmation emails
    - New release notifications

12. MULTI-LANGUAGE:
    - i18n support
    - Multiple currencies
    - Localized content

13. ACCESSIBILITY:
    - ARIA labels
    - Keyboard navigation
    - Screen reader support
    - High contrast mode

14. PERFORMANCE:
    - Image lazy loading
    - Code splitting
    - CDN for images
    - Service worker/PWA
    - Caching strategies

15. SEO:
    - Meta tags
    - Open Graph tags
    - Structured data (Schema.org)
    - Sitemap
    - Canonical URLs
*/

console.log('So many features to explore!');


## 21. Resources and Next Steps

Continue learning and building.


In [ ]:
/*
OFFICIAL DOCUMENTATION:

Shopify:
- Storefront API: https://shopify.dev/docs/api/storefront
- GraphQL Reference: https://shopify.dev/docs/api/storefront/latest
- JavaScript Buy SDK: https://shopify.github.io/js-buy-sdk/
- Shopify Partners: https://partners.shopify.com

React:
- React Docs: https://react.dev
- React Router: https://reactrouter.com
- Apollo Client: https://www.apollographql.com/docs/react/

LEARNING RESOURCES:
- Shopify Dev YouTube Channel
- Shopify Community Forums
- React + Shopify tutorials
- E-commerce best practices

TOOLS:
- Shopify GraphiQL Explorer: Test queries in your store admin
- Shopify CLI: Command-line tools for development
- Polaris: Shopify's design system

COMMUNITY:
- Shopify Community Forums
- Stack Overflow (shopify tag)
- Reddit r/shopify
- Shopify Discord

NEXT STEPS:
1. Add more books to your store
2. Customize the design
3. Add your branding
4. Implement advanced features
5. Test thoroughly
6. Deploy to production
7. Market your bookstore
8. Gather customer feedback
9. Iterate and improve

CONGRATULATIONS! 🎉
You now have a fully functional online bookstore powered by Shopify!

Key Benefits:
✓ Real payment processing through Shopify
✓ Inventory management handled by Shopify
✓ Secure checkout
✓ Order management in Shopify admin
✓ Customer data management
✓ Shipping and fulfillment tools
✓ Analytics and reporting
✓ Scalable infrastructure

Happy selling! 📚💰
*/

console.log('Your Shopify bookstore is ready to launch!');
